# Import libraries and dataset into environment

In [1]:
import dill
import os
import sys
import pandas as pd
import numpy as np
import xgboost as xgb
import math as mt
import shap
from sklearn.metrics import confusion_matrix, roc_curve, auc
from MLstatkit import Bootstrapping
import gc
from joblib import Parallel, delayed, externals
import multiprocessing
num_cores = multiprocessing.cpu_count() - 1
import defined_functions
from defined_functions import sum_metric, met_collate_func
defined_functions.pd = pd

In [2]:
path = os.getcwd()
sys.path.append(path)

save_file = os.path.join(path, "session.pkl")

with open(save_file, "rb") as f:
    state = dill.load(f)

split_list_valid_smote = state["split_list_valid_smote"]
split_list_valid_smote_final = state["split_list_valid_smote_final"]
split_list_fil_valid_smote_final = state["split_list_fil_valid_smote_final"]
split_list_sim_onset_valid_smote_final = state["split_list_sim_onset_valid_smote_final"]
split_list_vldiag_valid_smote_final = state["split_list_vldiag_valid_smote_final"]
split_list_diag1_valid_smote_final = state["split_list_diag1_valid_smote_final"]
split_list_diag2_valid_smote_final = state["split_list_diag2_valid_smote_final"]

# Define functions

In [3]:
# Define function to train XGBoost model
def run_single_xgboost_split(i, split):
    """Processes a single cross-validation data split loop iteration."""
    train_df = split['train'].copy()
    valid_df = split['valid'].copy()
    test_df = split['test'].copy()
    
    # Target label mapping ('Non severe' -> 0, 'Severe' -> 1)
    # Assumes categorical strings matching previous blocks
    label_map = {"Non severe": 0, "Severe": 1}
    y_train = train_df['outcome'].map(label_map).values
    y_valid = valid_df['outcome'].map(label_map).values
    y_test = test_df['outcome'].map(label_map).values

    X_train = train_df.drop(columns = ['outcome'])
    X_valid = valid_df.drop(columns = ['outcome'])
    X_test = test_df.drop(columns = ['outcome'])

    cat_cols = ['age', 'gender', 'vaccination', 'comorbidity']
    for col in cat_cols:
        X_train[col] = X_train[col].astype('category')
        X_valid[col] = X_valid[col].astype('category')
        X_test[col] = X_test[col].astype('category')

    dtrain = xgb.DMatrix(X_train, label = y_train, enable_categorical = True)
    dvalid = xgb.DMatrix(X_valid, label = y_valid, enable_categorical = True)
    dtest = xgb.DMatrix(X_test, label = y_test, enable_categorical = True)
    
   # Setting model parameters
    params = {
        'objective': 'binary:logistic',
        'eval_metric': ['aucpr'],
        'learning_rate': 0.003,    # Step size shrinkage used in update to prevents overfitting. (eta)
        'max_depth': 1,    # Maximum depth of a tree.
        'min_child_weight': 2,    # Minimum sum of instance weight (hessian) needed in a child.
        'subsample': 0.6,   # Subsample ratio of the training instances.
        'colsample_bytree': 0.6,    # Subsample ratio of columns when constructing each tree.
        'colsample_bynode': 0.6,    # Subsample ratio of columns when constructing each node.
        'colsample_bylevel': 0.6,    # Subsample ratio of columns when constructing each level.
        'gamma': 0.08, # Minimum loss reduction to make a further partition
        'lambda': 2.5,    # L2 regularization term on weights.
        'alpha': 0.5, # L1 regularization term on weights.
        'max_delta_step': 7,   # Maximum delta step we allow each leaf output to be.
        'scale_pos_weight': 1,  # Control the balance of positive and negative weights
        'tree_method': 'hist',
        'max_leaves': 2,  # Maximum number of nodes to be added
        'max_bin': 255, # Maximum number of discrete bins to bucket continuous features
        'grow_policy': 'depthwise', # Controls a way nodes are added to the tree
        'device': 'cuda',   # Utilizes GPU acceleration; If no GPU in device, please change to 'cpu' to enable model train on CPU.
        'verbosity': 0
    }
    
    # Setup model evaluation validation monitoring
    evals = [(dtrain, 'train'), (dvalid, 'validation')]
    
    # Train the XGBoost model
    xgb_model = xgb.train(
        params = params,
        dtrain = dtrain,
        num_boost_round = 1000,
        evals = evals,
        early_stopping_rounds = 200,
        verbose_eval = 100
    )
    
    # Make predictions on Testing Set
    xgb_pred_prob = xgb_model.predict(dtest)

    # Calculate Area Under the ROC curve
    fpr, tpr_curve, _ = roc_curve(y_test, xgb_pred_prob, pos_label = 1)
    auc_val_xgb = auc(fpr, tpr_curve)
    
    # Store standard structure dictionary to act like R's pROC container
    roc_container = {
        'actual': y_test,
        'probabilities': xgb_pred_prob
    }
    
    # Calculate Area Under the Precision-Recall Curve (AUPRC / PR-AUC)
    auprc_val, prc_ci_lower, prc_ci_upper = Bootstrapping(y_test, xgb_pred_prob, 'pr_auc')

    # Convert predicted probabilities to the labels
    xgb_pred = (xgb_pred_prob >= 0.5).astype(int)
    xgb_pred_res = np.where(xgb_pred == 0, "Non severe", "Severe")
    
    # Compute Confusion Matrix (Test)
    tn, fp, fn, tp = confusion_matrix(y_test, xgb_pred, labels = [0, 1]).ravel()
    
    # Format a formal R-styled evaluation matrix lookup dataframe 
    cfm_xgb = pd.DataFrame(
        [[tn, fp], [fn, tp]], 
        index = ["Non severe", "Severe"], 
        columns = ["Non severe", "Severe"]
    )
    cfm_xgb.index.name = 'Prediction'
    cfm_xgb.columns.name = 'Observed'
    
    # Calculate performance metrics
    accuracy_xgb = (tp + tn) / (tn + fp + fn + tp) if (tn + fp + fn + tp) > 0 else 0
    sensitivity_xgb = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity_xgb = tn / (tn + fp) if (tn + fp) > 0 else 0
    npv_xgb = tn / (tn + fn) if (tn + fn) > 0 else 0
    precision_xgb = tp / (tp + fp) if (tp + fp) > 0 else 0
    
    # Make predictions on Training Set to gather accuracy
    xgb_pred_train_prob = xgb_model.predict(dtrain)
    xgb_pred_train = (xgb_pred_train_prob >= 0.5).astype(int)
    tn_tr, fp_tr, fn_tr, tp_tr = confusion_matrix(y_train, xgb_pred_train, labels = [0, 1]).ravel()
    accuracy_xgb_train = (tp_tr + tn_tr) / (tn_tr + fp_tr + fn_tr + tp_tr)
     
    explainer = shap.TreeExplainer(model = xgb_model)
    shap_values = explainer(dtrain).values
    
    return {
        "model": xgb_model,
        "confusion_matrix": cfm_xgb,
        "accuracy_training": accuracy_xgb_train,
        "accuracy_testing": accuracy_xgb,
        "sensitivity": sensitivity_xgb,
        "specificity": specificity_xgb,
        "precision": precision_xgb,
        "npv": npv_xgb,
        "roc": roc_container,
        "AUC_value": auc_val_xgb,
        "PRC_val": auprc_val,
        "PRC_lower_ci": prc_ci_lower,
        "PRC_upper_ci": prc_ci_upper,
        "SHAP_values": shap_values,
        "prediction": xgb_pred_res,
        "pred_prob": xgb_pred_prob
    }

# Define function to train XGBoost for 100 times in parallel
def model_func_xgb_tune(data_list):
    """Main execution wrapper to distribute XGBoost tuning loops over system CPU threads."""
    # Count system resource availability profiles
    num_cores = multiprocessing.cpu_count() - 1
    
    if __name__ == '__main__':
        try:
            result_list = Parallel(n_jobs = num_cores)(
                delayed(run_single_xgboost_split)(i, data_list[i]) 
                for i in range(100)
                )
        finally:
            externals.loky.get_reusable_executor().shutdown(wait = True)
            gc.collect()

    return result_list

# Fitting dataset into the model

## Fitting data list without VL information

In [4]:
xgb_fil_list = model_func_xgb_tune(split_list_fil_valid_smote_final)
xgb_fil_met_summary = sum_metric(xgb_fil_list)
xgb_fil_metrics_summary = xgb_fil_met_summary["metric_summary"]
xgb_fil_summary = met_collate_func(xgb_fil_metrics_summary).assign(
    models = "XGBoost (No VL info & SMOTE)"
)
xgb_fil_summary

/home/blaw004/anaconda3/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [13:43:17] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  self.starting_round = model.num_boosted_rounds()
/home/blaw004/anaconda3/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [13:43:17] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  self.starting_round = model.num_boosted_rounds()
/home/blaw004/anaconda3/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [13:43:17] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  self.starting_round = model.num_boosted_rounds()
/home/blaw004/anaconda3/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [13:43:17] WARNING: /__w/xgboost/xgboost/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  self.star

[0]	train-aucpr:0.54305	validation-aucpr:0.04013[0]	train-aucpr:0.51998	validation-aucpr:0.03169[0]	train-aucpr:0.52290	validation-aucpr:0.03592


[0]	train-aucpr:0.51440	validation-aucpr:0.03697
[0]	train-aucpr:0.54171	validation-aucpr:0.03141
[100]	train-aucpr:0.89278	validation-aucpr:0.29217
[100]	train-aucpr:0.93296	validation-aucpr:0.26603
[100]	train-aucpr:0.91587	validation-aucpr:0.18920
[100]	train-aucpr:0.93666	validation-aucpr:0.20598
[0]	train-aucpr:0.52476	validation-aucpr:0.03936
[100]	train-aucpr:0.91765	validation-aucpr:0.24318
[200]	train-aucpr:0.89278	validation-aucpr:0.29217
[200]	train-aucpr:0.93194	validation-aucpr:0.26766
[0]	train-aucpr:0.56957	validation-aucpr:0.02647
[200]	train-aucpr:0.91367	validation-aucpr:0.16805
[205]	train-aucpr:0.91367	validation-aucpr:0.16805
[220]	train-aucpr:0.89278	validation-aucpr:0.29217
[222]	train-aucpr:0.93201	validation-aucpr:0.26766
[0]	train-aucpr:0.56935	validation-aucpr:0.04865
[100]	train-aucpr:0.90003	validation-aucpr:0.40

Bootstrapping pr_auc:   0%|          | 0/1000 [00:00<?, ?it/s]/home/blaw004/anaconda3/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [13:43:18] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  self.starting_round = model.num_boosted_rounds()
/home/blaw004/anaconda3/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [13:43:18] WARNING: /__w/xgboost/xgboost/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  self.starting_round = model.num_boosted_rounds()
Bootstrapping pr_auc:  16%|█▋        | 165/1000 [00:00<00:00, 1646.84it/s]/home/blaw004/anaconda3/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [13:43:18] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  self.starting_round = model.num_boosted_rounds()
/home/blaw004/anaconda3/lib/python3.12/site-packages/xgboost/callback.py:3


[500]	train-aucpr:0.92125	validation-aucpr:0.46352
[553]	train-aucpr:0.92137	validation-aucpr:0.44218
[200]	train-aucpr:0.90569	validation-aucpr:0.15497
[206]	train-aucpr:0.90569	validation-aucpr:0.15497
[0]	train-aucpr:0.57580	validation-aucpr:0.03009
[0]	train-aucpr:0.57182	validation-aucpr:0.03021
[100]	train-aucpr:0.92062	validation-aucpr:0.38131
[100]	train-aucpr:0.92213	validation-aucpr:0.19746
[200]	train-aucpr:0.92211	validation-aucpr:0.39427
[263]	train-aucpr:0.92062	validation-aucpr:0.38131
[200]	train-aucpr:0.92146	validation-aucpr:0.19904
[283]	train-aucpr:0.92146	validation-aucpr:0.19904


Bootstrapping pr_auc:  94%|█████████▎| 935/1000 [00:00<00:00, 1265.23it/s]

...

[0]	train-aucpr:0.54650	validation-aucpr:0.03183
[100]	train-aucpr:0.89924	validation-aucpr:0.29527
[100]	train-aucpr:0.89237	validation-aucpr:0.36024
[0]	train-aucpr:0.55065	validation-aucpr:0.03644
[200]	train-aucpr:0.89202	validation-aucpr:0.36024
[100]	train-aucpr:0.94531	validation-aucpr:0.28793
[200]	train-aucpr:0.89225	validation-aucpr:0.30980
[300]	train-aucpr:0.88913	validation-aucpr:0.36040
[0]	train-aucpr:0.57260	validation-aucpr:0.03198
[200]	train-aucpr:0.94531	validation-aucpr:0.28793
[367]	train-aucpr:0.88913	validation-aucpr:0.36040
[300]	train-aucpr:0.89532	validation-aucpr:0.29608
[218]	train-aucpr:0.94531	validation-aucpr:0.28793
[318]	train-aucpr:0.89532	validation-aucpr:0.29608
[0]	train-aucpr:0.58415	validation-aucpr:0.03021
[100]	train-aucpr:0.90429	validation-aucpr:0.41417


Bootstrapping pr_auc:  28%|██▊       | 283/1000 [00:00<00:00, 1452.69it/s]

[200]	train-aucpr:0.90209	validation-aucpr:0.41714
[0]	train-aucpr:0.50911	validation-aucpr:0.04052
[100]	train-aucpr:0.91049	validation-aucpr:0.34030
[300]	train-aucpr:0.90209	validation-aucpr:0.41714
[100]	train-aucpr:0.90319	validation-aucpr:0.24988
[345]	train-aucpr:0.90209	validation-aucpr:0.41714
[200]	train-aucpr:0.90757	validation-aucpr:0.33356
[221]	train-aucpr:0.91049	validation-aucpr:0.34030
[200]	train-aucpr:0.90343	validation-aucpr:0.25536
[203]	train-aucpr:0.90319	validation-aucpr:0.24988


Bootstrapping pr_auc: 100%|██████████| 1000/1000 [00:00<00:00, 1541.15it/s]


bound,Metrics,estimate,lower,upper,models
0,AUPRC_value,40.4812,22.3328,65.7593,XGBoost (No VL info & SMOTE)
1,AUROC_value,91.5237,80.6452,97.6636,XGBoost (No VL info & SMOTE)
2,Accuracy,87.6314,81.3822,93.0665,XGBoost (No VL info & SMOTE)
3,Accuracy_train,86.4616,81.2513,90.4322,XGBoost (No VL info & SMOTE)
4,NPV,99.3997,98.3108,100.0000,XGBoost (No VL info & SMOTE)
5,Precision,18.1561,12.3177,26.8013,XGBoost (No VL info & SMOTE)
6,Sensitivity,82.7000,50.0000,100.0000,XGBoost (No VL info & SMOTE)
7,Specificity,87.7850,80.9657,94.0810,XGBoost (No VL info & SMOTE)


## Fitting data list with simulated VL at symptom onset

In [5]:
xgb_vlsymp_sim_list = model_func_xgb_tune(split_list_sim_onset_valid_smote_final)
xgb_vlsymp_sim_met_summary = sum_metric(xgb_vlsymp_sim_list)
xgb_vlsymp_sim_metrics_summary = xgb_vlsymp_sim_met_summary["metric_summary"]
xgb_vlsymp_sim_summary = met_collate_func(xgb_vlsymp_sim_metrics_summary).assign(
    models = "XGBoost (VL symp simulated & SMOTE)"
)
xgb_vlsymp_sim_summary

/home/blaw004/anaconda3/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [13:43:27] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  self.starting_round = model.num_boosted_rounds()
/home/blaw004/anaconda3/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [13:43:27] WARNING: /__w/xgboost/xgboost/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  self.starting_round = model.num_boosted_rounds()
/home/blaw004/anaconda3/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [13:43:27] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  self.starting_round = model.num_boosted_rounds()
/home/blaw004/anaconda3/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [13:43:27] WARNING: /__w/xgboost/xgboost/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any 

[0]	train-aucpr:0.51440	validation-aucpr:0.03697
[0]	train-aucpr:0.54305	validation-aucpr:0.04013
[100]	train-aucpr:0.92000	validation-aucpr:0.17265
[0]	train-aucpr:0.57580	validation-aucpr:0.03009
[0]	train-aucpr:0.53639	validation-aucpr:0.03127
[0]	train-aucpr:0.54628	validation-aucpr:0.03734
[0]	train-aucpr:0.56935	validation-aucpr:0.04865
[200]	train-aucpr:0.91966	validation-aucpr:0.17626
[100]	train-aucpr:0.89948	validation-aucpr:0.30697
[0]	train-aucpr:0.52476	validation-aucpr:0.03936
[209]	train-aucpr:0.91966	validation-aucpr:0.17626
[0]	train-aucpr:0.57182	validation-aucpr:0.03021
[100]	train-aucpr:0.92566	validation-aucpr:0.39720
[100]	train-aucpr:0.94450	validation-aucpr:0.23491
[100]	train-aucpr:0.93525	validation-aucpr:0.22830
[200]	train-aucpr:0.89636	validation-aucpr:0.30800
[204]	train-aucpr:0.89780	validation-aucpr:0.30656
[100]	train-aucpr:0.94962	validation-aucpr:0.26426
[100]	train-aucpr:0.90286	validation-aucpr:0.45641
[0]	train-aucpr:0.54920	validation-aucpr:0.0298

Bootstrapping pr_auc:   0%|          | 0/1000 [00:00<?, ?it/s]/home/blaw004/anaconda3/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [13:43:27] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  self.starting_round = model.num_boosted_rounds()
/home/blaw004/anaconda3/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [13:43:27] WARNING: /__w/xgboost/xgboost/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  self.starting_round = model.num_boosted_rounds()
Bootstrapping pr_auc:   0%|          | 0/1000 [00:00<?, ?it/s]/home/blaw004/anaconda3/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [13:43:27] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  self.starting_round = model.num_boosted_rounds()
/home/blaw004/anaconda3/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarn

[200]	train-aucpr:0.94003	validation-aucpr:0.22034
[204]	train-aucpr:0.93924	validation-aucpr:0.22109
[0]	train-aucpr:0.55870	validation-aucpr:0.04176
[100]	train-aucpr:0.91904	validation-aucpr:0.18286
[0]	train-aucpr:0.56957	validation-aucpr:0.02647
[0]	train-aucpr:0.52290	validation-aucpr:0.03592
[100]	train-aucpr:0.92027	validation-aucpr:0.48612
[200]	train-aucpr:0.91824	validation-aucpr:0.17473
[100]	train-aucpr:0.91687	validation-aucpr:0.30202
[100]	train-aucpr:0.91895	validation-aucpr:0.39489
[200]	train-aucpr:0.92153	validation-aucpr:0.47576
[0]	train-aucpr:0.51998	validation-aucpr:0.03169
[300]	train-aucpr:0.91877	validation-aucpr:0.16766
[200]	train-aucpr:0.91652	validation-aucpr:0.30094
[317]	train-aucpr:0.92279	validation-aucpr:0.17071
[222]	train-aucpr:0.91609	validation-aucpr:0.29197
[200]	train-aucpr:0.92289	validation-aucpr:0.48152
[292]	train-aucpr:0.92441	validation-aucpr:0.47338
[100]	train-aucpr:0.93736	validation-aucpr:0.37773
[300]	train-aucpr:0.92381	validation-au

Bootstrapping pr_auc:  57%|█████▊    | 575/1000 [00:00<00:00, 1396.75it/s]

...

[226]	train-aucpr:0.95111	validation-aucpr:0.25502
[0]	train-aucpr:0.57260	validation-aucpr:0.03198
[100]	train-aucpr:0.90928	validation-aucpr:0.37116
[0]	train-aucpr:0.58415	validation-aucpr:0.03021
[200]	train-aucpr:0.90674	validation-aucpr:0.37434
[219]	train-aucpr:0.90798	validation-aucpr:0.35818
[0]	train-aucpr:0.50911	validation-aucpr:0.04052
[100]	train-aucpr:0.91287	validation-aucpr:0.30460
[100]	train-aucpr:0.90015	validation-aucpr:0.33743
[200]	train-aucpr:0.91285	validation-aucpr:0.27896
[207]	train-aucpr:0.91192	validation-aucpr:0.27681


Bootstrapping pr_auc:  16%|█▌        | 156/1000 [00:00<00:00, 1558.15it/s]

[200]	train-aucpr:0.90475	validation-aucpr:0.31945
[204]	train-aucpr:0.90485	validation-aucpr:0.31945


Bootstrapping pr_auc: 100%|██████████| 1000/1000 [00:00<00:00, 1512.72it/s]


bound,Metrics,estimate,lower,upper,models
0,AUPRC_value,39.6819,19.6116,61.3963,XGBoost (VL symp simulated & SMOTE)
1,AUROC_value,91.1760,79.7924,97.5389,XGBoost (VL symp simulated & SMOTE)
2,Accuracy,88.8338,83.9879,93.5272,XGBoost (VL symp simulated & SMOTE)
3,Accuracy_train,86.3551,81.6965,90.4102,XGBoost (VL symp simulated & SMOTE)
4,NPV,99.2729,98.3301,100.0000,XGBoost (VL symp simulated & SMOTE)
5,Precision,19.0760,13.6383,29.1028,XGBoost (VL symp simulated & SMOTE)
6,Sensitivity,78.8000,50.0000,100.0000,XGBoost (VL symp simulated & SMOTE)
7,Specificity,89.1464,84.1121,94.2601,XGBoost (VL symp simulated & SMOTE)


## Fitting data list with VL at diagnosis

In [6]:
xgb_vldiag_list = model_func_xgb_tune(split_list_vldiag_valid_smote_final)
xgb_vldiag_met_summary = sum_metric(xgb_vldiag_list)
xgb_vldiag_metrics_summary = xgb_vldiag_met_summary["metric_summary"]
xgb_vldiag_summary = met_collate_func(xgb_vldiag_metrics_summary).assign(
    models = "XGBoost (VL diag & SMOTE)"
)
xgb_vldiag_summary

/home/blaw004/anaconda3/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [13:43:37] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  self.starting_round = model.num_boosted_rounds()
/home/blaw004/anaconda3/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [13:43:37] WARNING: /__w/xgboost/xgboost/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  self.starting_round = model.num_boosted_rounds()
/home/blaw004/anaconda3/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [13:43:37] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  self.starting_round = model.num_boosted_rounds()
/home/blaw004/anaconda3/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [13:43:37] WARNING: /__w/xgboost/xgboost/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any 

[0]	train-aucpr:0.54171	validation-aucpr:0.03141
[0]	train-aucpr:0.54305	validation-aucpr:0.04013
[100]	train-aucpr:0.94442	validation-aucpr:0.30630
[100]	train-aucpr:0.91024	validation-aucpr:0.32457
[0]	train-aucpr:0.53639	validation-aucpr:0.03127
[200]	train-aucpr:0.90878	validation-aucpr:0.34234
[204]	train-aucpr:0.90801	validation-aucpr:0.34234
[0]	train-aucpr:0.56935	validation-aucpr:0.04865
[200]	train-aucpr:0.94454	validation-aucpr:0.31688
[0]	train-aucpr:0.51440	validation-aucpr:0.03697
[100]	train-aucpr:0.93949	validation-aucpr:0.27505
[100]	train-aucpr:0.95642	validation-aucpr:0.33776
[300]	train-aucpr:0.94414	validation-aucpr:0.31591
[100]	train-aucpr:0.93029	validation-aucpr:0.32423
[200]	train-aucpr:0.93941	validation-aucpr:0.28690
[204]	train-aucpr:0.94024	validation-aucpr:0.28952
[200]	train-aucpr:0.95550	validation-aucpr:0.31922
[0]	train-aucpr:0.51998	validation-aucpr:0.03169
[400]	train-aucpr:0.94528	validation-aucpr:0.31600
[200]	train-aucpr:0.92637	validation-aucpr:

/home/blaw004/anaconda3/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [13:43:37] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  self.starting_round = model.num_boosted_rounds()
/home/blaw004/anaconda3/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [13:43:37] WARNING: /__w/xgboost/xgboost/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  self.starting_round = model.num_boosted_rounds()
/home/blaw004/anaconda3/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [13:43:37] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  self.starting_round = model.num_boosted_rounds()
/home/blaw004/anaconda3/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [13:43:37] WARNING: /__w/xgboost/xgboost/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any 

[0]	train-aucpr:0.57580	validation-aucpr:0.03009
[100]	train-aucpr:0.94885	validation-aucpr:0.23028
[300]	train-aucpr:0.93924	validation-aucpr:0.61618
[0]	train-aucpr:0.54920	validation-aucpr:0.02984
[0]	train-aucpr:0.55870	validation-aucpr:0.04176
[200]	train-aucpr:0.91105	validation-aucpr:0.49585
[100]	train-aucpr:0.93784	validation-aucpr:0.30242
[200]	train-aucpr:0.94809	validation-aucpr:0.21827
[204]	train-aucpr:0.94807	validation-aucpr:0.21827
[100]	train-aucpr:0.91950	validation-aucpr:0.34320
[100]	train-aucpr:0.92271	validation-aucpr:0.51840
[300]	train-aucpr:0.91540	validation-aucpr:0.46737
[400]	train-aucpr:0.94176	validation-aucpr:0.61596
[335]	train-aucpr:0.91672	validation-aucpr:0.46997
[200]	train-aucpr:0.93794	validation-aucpr:0.30780
[227]	train-aucpr:0.93752	validation-aucpr:0.30883
[200]	train-aucpr:0.92006	validation-aucpr:0.34864
[205]	train-aucpr:0.91983	validation-aucpr:0.34889
[200]	train-aucpr:0.92371	validation-aucpr:0.51341
[0]	train-aucpr:0.56957	validation-au

Bootstrapping pr_auc:  16%|█▌        | 162/1000 [00:00<00:00, 1611.51it/s]/home/blaw004/anaconda3/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [13:43:38] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  self.starting_round = model.num_boosted_rounds()
/home/blaw004/anaconda3/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [13:43:38] WARNING: /__w/xgboost/xgboost/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  self.starting_round = model.num_boosted_rounds()
Bootstrapping pr_auc:  45%|████▌     | 453/1000 [00:00<00:00, 1391.82it/s]

...

[200]	train-aucpr:0.91089	validation-aucpr:0.35435
[200]	train-aucpr:0.90193	validation-aucpr:0.49106
[100]	train-aucpr:0.95714	validation-aucpr:0.31877
[0]	train-aucpr:0.57260	validation-aucpr:0.03198
[400]	train-aucpr:0.89972	validation-aucpr:0.38135
[300]	train-aucpr:0.90343	validation-aucpr:0.47895
[200]	train-aucpr:0.95687	validation-aucpr:0.31144
[300]	train-aucpr:0.91106	validation-aucpr:0.34931
[0]	train-aucpr:0.58415	validation-aucpr:0.03021
[238]	train-aucpr:0.95685	validation-aucpr:0.31144
[450]	train-aucpr:0.90084	validation-aucpr:0.38204
[400]	train-aucpr:0.90464	validation-aucpr:0.49552
[100]	train-aucpr:0.92130	validation-aucpr:0.41665
[400]	train-aucpr:0.91745	validation-aucpr:0.39390
[500]	train-aucpr:0.90486	validation-aucpr:0.48354
[100]	train-aucpr:0.92784	validation-aucpr:0.37425
[527]	train-aucpr:0.90502	validation-aucpr:0.48757
[500]	train-aucpr:0.91782	validation-aucpr:0.39497
[200]	train-aucpr:0.92248	validation-aucpr:0.42378
[206]	train-aucpr:0.92298	validatio

Bootstrapping pr_auc:  72%|███████▎  | 725/1000 [00:00<00:00, 1385.95it/s]

[800]	train-aucpr:0.91942	validation-aucpr:0.40389
[821]	train-aucpr:0.91936	validation-aucpr:0.40274
[0]	train-aucpr:0.50911	validation-aucpr:0.04052
[100]	train-aucpr:0.90924	validation-aucpr:0.34841
[200]	train-aucpr:0.91326	validation-aucpr:0.32581
[204]	train-aucpr:0.91341	validation-aucpr:0.32581


Bootstrapping pr_auc: 100%|██████████| 1000/1000 [00:00<00:00, 1630.76it/s]


bound,Metrics,estimate,lower,upper,models
0,AUPRC_value,43.5403,24.7038,64.8809,XGBoost (VL diag & SMOTE)
1,AUROC_value,92.8433,84.2551,98.2773,XGBoost (VL diag & SMOTE)
2,Accuracy,89.8429,84.7356,93.6556,XGBoost (VL diag & SMOTE)
3,Accuracy_train,86.3126,81.7952,91.4901,XGBoost (VL diag & SMOTE)
4,NPV,99.3111,98.6506,100.0000,XGBoost (VL diag & SMOTE)
5,Precision,20.8185,14.0774,30.5478,XGBoost (VL diag & SMOTE)
6,Sensitivity,79.8000,60.0000,100.0000,XGBoost (VL diag & SMOTE)
7,Specificity,90.1558,84.8676,94.0966,XGBoost (VL diag & SMOTE)


## Fitting data list with VL at diagnosis & VL at 1-day after diagnosis

In [7]:
xgb_add1_list = model_func_xgb_tune(split_list_diag1_valid_smote_final)
xgb_add1_met_summary = sum_metric(xgb_add1_list)
xgb_add1_metrics_summary = xgb_add1_met_summary["metric_summary"]
xgb_add1_summary = met_collate_func(xgb_add1_metrics_summary).assign(
    models = "XGBoost (VL diag + 1 & SMOTE)"
)
xgb_add1_summary

/home/blaw004/anaconda3/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [13:43:48] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  self.starting_round = model.num_boosted_rounds()
/home/blaw004/anaconda3/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [13:43:48] WARNING: /__w/xgboost/xgboost/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  self.starting_round = model.num_boosted_rounds()
/home/blaw004/anaconda3/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [13:43:48] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  self.starting_round = model.num_boosted_rounds()
/home/blaw004/anaconda3/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [13:43:48] WARNING: /__w/xgboost/xgboost/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any 

[0]	train-aucpr:0.54305	validation-aucpr:0.04013
[0]	train-aucpr:0.52476	validation-aucpr:0.03936
[0]	train-aucpr:0.51998	validation-aucpr:0.03169
[0]	train-aucpr:0.53639	validation-aucpr:0.03127
[0]	train-aucpr:0.54628	validation-aucpr:0.03734
[100]	train-aucpr:0.90460	validation-aucpr:0.38568
[100]	train-aucpr:0.90632	validation-aucpr:0.54510
[0]	train-aucpr:0.54171	validation-aucpr:0.03141
[100]	train-aucpr:0.92694	validation-aucpr:0.60456
[100]	train-aucpr:0.93936	validation-aucpr:0.35165
[100]	train-aucpr:0.95322	validation-aucpr:0.18642
[200]	train-aucpr:0.90942	validation-aucpr:0.39777
[0]	train-aucpr:0.57182	validation-aucpr:0.03021
[200]	train-aucpr:0.91296	validation-aucpr:0.57822
[100]	train-aucpr:0.93838	validation-aucpr:0.36408
[0]	train-aucpr:0.57580	validation-aucpr:0.03009
[200]	train-aucpr:0.93531	validation-aucpr:0.59167
[200]	train-aucpr:0.94332	validation-aucpr:0.35745
[200]	train-aucpr:0.95561	validation-aucpr:0.19708
[242]	train-aucpr:0.94073	validation-aucpr:0.36

Bootstrapping pr_auc:  16%|█▌        | 155/1000 [00:00<00:00, 1544.98it/s]

[200]	train-aucpr:0.92004	validation-aucpr:0.21576
[300]	train-aucpr:0.94518	validation-aucpr:0.25398
[345]	train-aucpr:0.94599	validation-aucpr:0.25027
[300]	train-aucpr:0.92956	validation-aucpr:0.37344
[302]	train-aucpr:0.92964	validation-aucpr:0.37344
[200]	train-aucpr:0.92384	validation-aucpr:0.32855
[300]	train-aucpr:0.92549	validation-aucpr:0.48902
[200]	train-aucpr:0.92204	validation-aucpr:0.47490
[300]	train-aucpr:0.92321	validation-aucpr:0.22060
[302]	train-aucpr:0.92318	validation-aucpr:0.22004
[300]	train-aucpr:0.92340	validation-aucpr:0.32849
[300]	train-aucpr:0.92326	validation-aucpr:0.45852
[400]	train-aucpr:0.92973	validation-aucpr:0.48789
[430]	train-aucpr:0.92929	validation-aucpr:0.48752
[353]	train-aucpr:0.92565	validation-aucpr:0.34242
[360]	train-aucpr:0.92563	validation-aucpr:0.46370


Bootstrapping pr_auc: 100%|██████████| 1000/1000 [00:00<00:00, 1393.21it/s]


...

[100]	train-aucpr:0.90226	validation-aucpr:0.43138
[200]	train-aucpr:0.93584	validation-aucpr:0.34242
[300]	train-aucpr:0.93589	validation-aucpr:0.34186
[200]	train-aucpr:0.91231	validation-aucpr:0.41296
[300]	train-aucpr:0.92325	validation-aucpr:0.45237
[400]	train-aucpr:0.93958	validation-aucpr:0.34693
[300]	train-aucpr:0.91601	validation-aucpr:0.38436
[307]	train-aucpr:0.91581	validation-aucpr:0.38336
[400]	train-aucpr:0.92574	validation-aucpr:0.47240
[500]	train-aucpr:0.93977	validation-aucpr:0.46517
[600]	train-aucpr:0.94449	validation-aucpr:0.46728
[500]	train-aucpr:0.92593	validation-aucpr:0.46942
[700]	train-aucpr:0.94490	validation-aucpr:0.46679
[589]	train-aucpr:0.92659	validation-aucpr:0.46938


Bootstrapping pr_auc:  58%|█████▊    | 580/1000 [00:00<00:00, 1304.67it/s]

[778]	train-aucpr:0.94507	validation-aucpr:0.46747


Bootstrapping pr_auc: 100%|██████████| 1000/1000 [00:00<00:00, 1525.53it/s]


bound,Metrics,estimate,lower,upper,models
0,AUPRC_value,43.3939,21.7415,65.9645,XGBoost (VL diag + 1 & SMOTE)
1,AUROC_value,93.2148,85.7114,98.2488,XGBoost (VL diag + 1 & SMOTE)
2,Accuracy,87.4743,82.6208,91.5408,XGBoost (VL diag + 1 & SMOTE)
3,Accuracy_train,86.5997,82.3196,90.7360,XGBoost (VL diag + 1 & SMOTE)
4,NPV,99.3785,98.4374,100.0000,XGBoost (VL diag + 1 & SMOTE)
5,Precision,17.5414,12.7716,24.3015,XGBoost (VL diag + 1 & SMOTE)
6,Sensitivity,82.3000,54.7500,100.0000,XGBoost (VL diag + 1 & SMOTE)
7,Specificity,87.6355,82.5389,91.6044,XGBoost (VL diag + 1 & SMOTE)


## Fitting data list with VL at diagnosis & VL at 2-days after diagnosis

In [8]:
xgb_add2_list = model_func_xgb_tune(split_list_diag2_valid_smote_final)
xgb_add2_met_summary = sum_metric(xgb_add2_list)
xgb_add2_metrics_summary = xgb_add2_met_summary["metric_summary"]
xgb_add2_summary = met_collate_func(xgb_add2_metrics_summary).assign(
    models = "XGBoost (VL diag + 2 & SMOTE)"
)
xgb_add2_summary

/home/blaw004/anaconda3/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [13:43:58] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  self.starting_round = model.num_boosted_rounds()
/home/blaw004/anaconda3/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [13:43:58] WARNING: /__w/xgboost/xgboost/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  self.starting_round = model.num_boosted_rounds()
/home/blaw004/anaconda3/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [13:43:58] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  self.starting_round = model.num_boosted_rounds()
/home/blaw004/anaconda3/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [13:43:58] WARNING: /__w/xgboost/xgboost/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any 

[0]	train-aucpr:0.52476	validation-aucpr:0.03936
[0]	train-aucpr:0.55870	validation-aucpr:0.04176
[100]	train-aucpr:0.91080	validation-aucpr:0.61483
[100]	train-aucpr:0.93049	validation-aucpr:0.54473
[0]	train-aucpr:0.51440	validation-aucpr:0.03697
[200]	train-aucpr:0.91759	validation-aucpr:0.58689
[0]	train-aucpr:0.54171	validation-aucpr:0.03141
[200]	train-aucpr:0.93273	validation-aucpr:0.54234
[100]	train-aucpr:0.92758	validation-aucpr:0.43148
[100]	train-aucpr:0.94085	validation-aucpr:0.45163
[300]	train-aucpr:0.91896	validation-aucpr:0.58023
[0]	train-aucpr:0.57115	validation-aucpr:0.02235
[273]	train-aucpr:0.93347	validation-aucpr:0.54254
[339]	train-aucpr:0.91992	validation-aucpr:0.57184
[0]	train-aucpr:0.57182	validation-aucpr:0.03021
[200]	train-aucpr:0.93164	validation-aucpr:0.37747
[0]	train-aucpr:0.57580	validation-aucpr:0.03009
[0]	train-aucpr:0.56957	validation-aucpr:0.02647
[200]	train-aucpr:0.94398	validation-aucpr:0.44019
[100]	train-aucpr:0.91982	validation-aucpr:0.35

/home/blaw004/anaconda3/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [13:43:58] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  self.starting_round = model.num_boosted_rounds()
/home/blaw004/anaconda3/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [13:43:58] WARNING: /__w/xgboost/xgboost/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  self.starting_round = model.num_boosted_rounds()
Bootstrapping pr_auc:   0%|          | 0/1000 [00:00<?, ?it/s]/home/blaw004/anaconda3/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [13:43:58] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  self.starting_round = model.num_boosted_rounds()
/home/blaw004/anaconda3/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [13:43:58] WARNING: /__w/xgboost/xgboost/src/context.cc:2

[800]	train-aucpr:0.94720	validation-aucpr:0.26463
[852]	train-aucpr:0.94745	validation-aucpr:0.26357
[300]	train-aucpr:0.95896	validation-aucpr:0.21136
[344]	train-aucpr:0.95875	validation-aucpr:0.21435


Bootstrapping pr_auc: 100%|██████████| 1000/1000 [00:00<00:00, 1358.84it/s]


...

[100]	train-aucpr:0.92832	validation-aucpr:0.46412
[400]	train-aucpr:0.96977	validation-aucpr:0.29674
[0]	train-aucpr:0.58415	validation-aucpr:0.03021
[0]	train-aucpr:0.50911	validation-aucpr:0.04052
[200]	train-aucpr:0.92894	validation-aucpr:0.54588
[500]	train-aucpr:0.96986	validation-aucpr:0.30120
[100]	train-aucpr:0.93832	validation-aucpr:0.33377
[514]	train-aucpr:0.96988	validation-aucpr:0.29942
[100]	train-aucpr:0.90668	validation-aucpr:0.44238
[300]	train-aucpr:0.92950	validation-aucpr:0.53730
[200]	train-aucpr:0.94096	validation-aucpr:0.34066
[200]	train-aucpr:0.91507	validation-aucpr:0.41096
[400]	train-aucpr:0.93150	validation-aucpr:0.55009
[300]	train-aucpr:0.94039	validation-aucpr:0.33992
[300]	train-aucpr:0.91835	validation-aucpr:0.39027
[304]	train-aucpr:0.91821	validation-aucpr:0.38648
[317]	train-aucpr:0.94038	validation-aucpr:0.34114
[500]	train-aucpr:0.93187	validation-aucpr:0.54216


Bootstrapping pr_auc:  94%|█████████▍| 940/1000 [00:00<00:00, 1254.98it/s]

[594]	train-aucpr:0.93206	validation-aucpr:0.54560


Bootstrapping pr_auc: 100%|██████████| 1000/1000 [00:00<00:00, 1456.31it/s]


bound,Metrics,estimate,lower,upper,models
0,AUPRC_value,44.7082,22.3797,63.8181,XGBoost (VL diag + 2 & SMOTE)
1,AUROC_value,93.6593,87.3107,98.2103,XGBoost (VL diag + 2 & SMOTE)
2,Accuracy,88.6979,84.5921,92.1450,XGBoost (VL diag + 2 & SMOTE)
3,Accuracy_train,87.2825,83.2269,91.4901,XGBoost (VL diag + 2 & SMOTE)
4,NPV,99.3846,98.3271,100.0000,XGBoost (VL diag + 2 & SMOTE)
5,Precision,19.1441,13.5135,25.7627,XGBoost (VL diag + 2 & SMOTE)
6,Sensitivity,82.2000,50.0000,100.0000,XGBoost (VL diag + 2 & SMOTE)
7,Specificity,88.9003,84.4237,92.8349,XGBoost (VL diag + 2 & SMOTE)


# Save model trained

In [9]:
path = os.getcwd()

state = {
    "xgb_fil_list": xgb_fil_list,
    "xgb_vlsymp_sim_list": xgb_vlsymp_sim_list,
    "xgb_vldiag_list": xgb_vldiag_list,
    "xgb_add1_list": xgb_add1_list,
    "xgb_add2_list": xgb_add2_list
}

save_file = os.path.join(path, "xgb_trained.pkl")

with open(save_file, "wb") as f:
    dill.dump(state, f)

#print(f"Saved to: {save_file}")

In [10]:
path = os.getcwd()

state = {
    "xgb_fil_summary": xgb_fil_summary,
    "xgb_vlsymp_sim_summary": xgb_vlsymp_sim_summary,
    "xgb_vldiag_summary": xgb_vldiag_summary,
    "xgb_add1_summary": xgb_add1_summary,
    "xgb_add2_summary": xgb_add2_summary
}

save_file = os.path.join(path, "xgb_metric_summary.pkl")

with open(save_file, "wb") as f:
    dill.dump(state, f)

#print(f"Saved to: {save_file}")